# Load the model

In [ ]:
%%capture
!pip install pip3-autoremove
!pip-autoremove torch torchvision torchaudio -y
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install --upgrade --no-cache-dir transformers

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.6.2: Fast Llama patching. Transformers: 4.52.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.6.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


# Load dataset

In [ ]:
import json

with open('/content/drive/MyDrive/stigma_project/data/healthunlocked-data/sft_sample_10000.json', 'r') as f:
  data = json.loads(f.read())
  f.close()

In [ ]:
dataset = []
for d in data:
  chat = {'chosen': [], 'rejected': []}
  chat['chosen'].append({'role': 'system',
               'content': "You are an advanced text analysis AI designed to identify and label social media content for any signs of stigmatized language or messaging regarding people with diabetes. You have expertise in understanding bias, stereotypes, and stigmatizing terms, and you apply your knowledge impartially. When presented with a social media post, you will classify the content based on whether it contains: 1. **Stigmatized Content**: Language that is discriminatory, biased, shaming, or marginalizing towards individuals with diabetes. 2. **Neutral Content**: Language that is neutral, factual, or supportive without stigmatizing. For each post, you must provide: 1. **Label**: 'Stigmatized Content' or 'Neutral Content.' 2. **Explanation**: A brief rationale for why the post was labeled this way. Your decisions must be based on the specific words and tone of the post, avoiding personal interpretation or assumptions about the author's intent unless explicitly stated in the text."})

  chat['chosen'].append({'role': 'user',
               'content': f"Here is a social media post. Please classify it as either 'Stigmatized Content' or 'Neutral Content' based on its language and tone regarding people with diabetes. **Post main body**: {d['post']}"})

  chat['chosen'].append({'role': 'assistant',
               'content': f"This post includes {d['label']} content."})

  op_label = 'neutral' if d['label'] == 'stigma' else 'stigma'

  chat['rejected'].append({'role': 'system',
               'content': "You are an advanced text analysis AI designed to identify and label social media content for any signs of stigmatized language or messaging regarding people with diabetes. You have expertise in understanding bias, stereotypes, and stigmatizing terms, and you apply your knowledge impartially. When presented with a social media post, you will classify the content based on whether it contains: 1. **Stigmatized Content**: Language that is discriminatory, biased, shaming, or marginalizing towards individuals with diabetes. 2. **Neutral Content**: Language that is neutral, factual, or supportive without stigmatizing. For each post, you must provide: 1. **Label**: 'Stigmatized Content' or 'Neutral Content.' 2. **Explanation**: A brief rationale for why the post was labeled this way. Your decisions must be based on the specific words and tone of the post, avoiding personal interpretation or assumptions about the author's intent unless explicitly stated in the text."})

  chat['rejected'].append({'role': 'user',
               'content': f"Here is a social media post. Please classify it as either 'Stigmatized Content' or 'Neutral Content' based on its language and tone regarding people with diabetes. **Post main body**: {d['post']}"})

  chat['rejected'].append({'role': 'assistant',
               'content': f"This post includes {op_label} content."})
  dataset.append(chat)

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(dataset)
print(dataset)

Dataset({
    features: ['chosen', 'rejected'],
    num_rows: 10000
})


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["chosen"]
    chosen = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    convos = examples["rejected"]
    rejected = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]

    return {"chosen": chosen, "rejected": rejected}

In [ ]:
dataset2 = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

# Train the model

In [ ]:
# Enable reward modelling stats
from unsloth import PatchDPOTrainer
PatchDPOTrainer()
from transformers import TrainingArguments
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 1,
        learning_rate = 5e-6,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
    beta = 0.1,
    train_dataset = dataset2,
    tokenizer = tokenizer,
    max_length = max_seq_length,
)

Extracting prompt in train dataset (num_proc=2):   0%|          | 0/10000 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=2):   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
tokenizer.decode(dpo_trainer.train_dataset[5]["rejected_input_ids"])

' neutral content.<|eot_id|><|eot_id|>'

In [ ]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
3.441 GB of memory reserved.


In [ ]:
trainer_stats = dpo_trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 1,250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856/3,000,000,000 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss,aux_loss
1,0.693100,0.000000,0.000000,0.000000,0.000000,-27.451767,-33.830505,0.861630,0.588151,0,0,0,0
2,0.693100,0.000000,0.000000,0.000000,0.000000,-24.655098,-33.745502,0.821032,0.626170,No Log,No Log,No Log,No Log
3,0.693300,0.000911,0.001154,0.500000,-0.000243,-27.203308,-35.020576,0.714213,0.393671,No Log,No Log,No Log,No Log
4,0.693600,0.000318,0.001294,0.250000,-0.000976,-23.743158,-33.110561,0.924478,0.603750,No Log,No Log,No Log,No Log
5,0.693300,-0.000788,-0.000443,0.500000,-0.000345,-26.376587,-36.038532,0.899054,0.676171,No Log,No Log,No Log,No Log
6,0.692500,0.000780,-0.000451,0.750000,0.001231,-26.004784,-34.038742,0.933327,0.550696,No Log,No Log,No Log,No Log
7,0.693200,0.000249,0.000330,0.500000,-0.000082,-25.341694,-33.762581,0.812047,0.762776,No Log,No Log,No Log,No Log
8,0.692800,0.000284,-0.000497,0.625000,0.000781,-24.599743,-34.920681,0.912419,0.519774,No Log,No Log,No Log,No Log
9,0.692800,0.000701,-0.000085,0.625000,0.000785,-25.706238,-34.268860,0.903868,0.688750,No Log,No Log,No Log,No Log
10,0.693200,-0.000924,-0.000762,0.500000,-0.000162,-25.162867,-33.840237,0.895090,0.680655,No Log,No Log,No Log,No Log


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss,aux_loss
1,0.693100,0.000000,0.000000,0.000000,0.000000,-27.451767,-33.830505,0.861630,0.588151,0,0,0,0
2,0.693100,0.000000,0.000000,0.000000,0.000000,-24.655098,-33.745502,0.821032,0.626170,No Log,No Log,No Log,No Log
3,0.693300,0.000911,0.001154,0.500000,-0.000243,-27.203308,-35.020576,0.714213,0.393671,No Log,No Log,No Log,No Log
4,0.693600,0.000318,0.001294,0.250000,-0.000976,-23.743158,-33.110561,0.924478,0.603750,No Log,No Log,No Log,No Log
5,0.693300,-0.000788,-0.000443,0.500000,-0.000345,-26.376587,-36.038532,0.899054,0.676171,No Log,No Log,No Log,No Log
6,0.692500,0.000780,-0.000451,0.750000,0.001231,-26.004784,-34.038742,0.933327,0.550696,No Log,No Log,No Log,No Log
7,0.693200,0.000249,0.000330,0.500000,-0.000082,-25.341694,-33.762581,0.812047,0.762776,No Log,No Log,No Log,No Log
8,0.692800,0.000284,-0.000497,0.625000,0.000781,-24.599743,-34.920681,0.912419,0.519774,No Log,No Log,No Log,No Log
9,0.692800,0.000701,-0.000085,0.625000,0.000785,-25.706238,-34.268860,0.903868,0.688750,No Log,No Log,No Log,No Log
10,0.693200,-0.000924,-0.000762,0.500000,-0.000162,-25.162867,-33.840237,0.895090,0.680655,No Log,No Log,No Log,No Log


KeyboardInterrupt: 

# Inference

In [ ]:
messages = [
    {'role': 'system',
               'content': "You are an advanced text analysis AI designed to identify and label social media content for any signs of stigmatized language or messaging regarding people with diabetes. You have expertise in understanding bias, stereotypes, and stigmatizing terms, and you apply your knowledge impartially. When presented with a social media post, you will classify the content based on whether it contains: 1. **Stigmatized Content**: Language that is discriminatory, biased, shaming, or marginalizing towards individuals with diabetes. 2. **Neutral Content**: Language that is neutral, factual, or supportive without stigmatizing. For each post, you must provide: 1. **Label**: 'Stigmatized Content' or 'Neutral Content.' 2. **Explanation**: A brief rationale for why the post was labeled this way. Your decisions must be based on the specific words and tone of the post, avoiding personal interpretation or assumptions about the author's intent unless explicitly stated in the text."},
    {'role': 'user',
               'content': """Here is a social media post. Please classify it as either 'Stigmatized Content' or 'Neutral Content' based on its language and tone regarding people with diabetes. **Post main body**: Tell us the crap you're dealing with this week. Did someone suggest cinnamon again? What about that relative who tried to pray the beetus away?

 As always, please keep in mind [our rules](https://www.reddit.com/r/diabetes/about/rules)"""},
]

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference


inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True,
                         temperature = 1.5, min_p = 0.1)
tokenizer.batch_decode(outputs)

["<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\nYou are an advanced text analysis AI designed to identify and label social media content for any signs of stigmatized language or messaging regarding people with diabetes. You have expertise in understanding bias, stereotypes, and stigmatizing terms, and you apply your knowledge impartially. When presented with a social media post, you will classify the content based on whether it contains: 1. **Stigmatized Content**: Language that is discriminatory, biased, shaming, or marginalizing towards individuals with diabetes. 2. **Neutral Content**: Language that is neutral, factual, or supportive without stigmatizing. For each post, you must provide: 1. **Label**: 'Stigmatized Content' or 'Neutral Content.' 2. **Explanation**: A brief rationale for why the post was labeled this way. Your decisions must be based on the specific words and tone of the post, avoiding

In [ ]:
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

**Label:** Neutral Content

**Explanation:** The post primarily focuses on the frustrations and experiences of the author dealing with people's misconceptions and uninformed comments about diabetes. While the comments mentioned, such as the "beetus away" joke, may be perceived as stigmatizing or mocking, they are framed within the context of the post's overall discussion about the author's dealing with those types of comments. The author is presenting this as their experience and not endorsing or condoning such behavior. Additionally, the post includes a link to the rules section of a relevant community, which likely addresses these issues and encourages respectful discussion. Overall, the
